### PTB-XL Predictive Model Project
### DS 5110 - 26F Data Eng II: Big Data Systems
-----
TO DO list: 
* import data
* JOIN
* data preprocessing and EDA
* start Pipeline architecture
* run baseline model(s) (logreg, L2, L1, etc.)


#### I. Import packages, start SparkSession
---

In [1]:
# packages
from pyspark.sql import SparkSession 

from pyspark.ml import Pipeline
from pyspark.ml.feature import *
from pyspark.ml.classification import LogisticRegression


In [2]:
# start SparkSession; set base paths

spark = SparkSession.builder.getOrCreate()

base_xlplus = "/standard/ds7200-apt4c/team2/ptb-xl-a-comprehensive-electrocardiographic-feature-dataset-1.0.1/features"
base_xl = "/standard/ds7200-apt4c/team2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/apps/software/standard/core/pyspark/4.2.0-py3.14/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/28 14:29:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
base_team2 = "/standard/ds7200-apt4c/team2"

#### II. Import data files
----
**Note:** `spark.read.csv()` can accept multiple files as a list of paths but it will stack them in the same DataFrame -- not ideal when columns are different

**Note:** `12sl`, `ecgdeli`, and `unig` are separate feature sets from different algorithms.  One row per ECG. 

In [4]:
# read 4 files & verify existence of JOIN key ecg_id
# (1) ptbxl_database.csv
# (2) 12sl_features.csv
# (3) ecgdeli_features.csv
# (4) unig_features.csv 

ptbxl = spark.read.csv(f"{base_xl}/ptbxl_database.csv", header=True, inferSchema=True)
"ecg_id" in ptbxl.columns

True

In [5]:
sl12 = spark.read.csv(f"{base_xlplus}/12sl_features.csv", header=True, inferSchema=True)  # Py variable cannot begin with #
"ecg_id" in sl12.columns

True

In [6]:
ecgdeli = spark.read.csv(f"{base_xlplus}/ecgdeli_features.csv", header=True, inferSchema=True)
"ecg_id" in ecgdeli.columns

True

In [7]:
unig = spark.read.csv(f"{base_xlplus}/unig_features.csv", header=True, inferSchema=True)
"ecg_id" in unig.columns

True

#### III. Select columns and JOIN datasets
---

In [8]:
# get a list of columns for each dataset

In [9]:
sl12.printSchema()

root
 |-- P_Area_I: double (nullable = true)
 |-- P_PeakTime_I: double (nullable = true)
 |-- Q_Area_I: double (nullable = true)
 |-- Q_PeakTime_I: double (nullable = true)
 |-- R_Area_I: double (nullable = true)
 |-- R_PeakTime_I: double (nullable = true)
 |-- S_Area_I: double (nullable = true)
 |-- S_PeakTime_I: double (nullable = true)
 |-- QRS_Balance_I: double (nullable = true)
 |-- T_Area_I: double (nullable = true)
 |-- T_PeakTime_I: double (nullable = true)
 |-- QRS_Area_I: double (nullable = true)
 |-- P_Area_II: double (nullable = true)
 |-- P_PeakTime_II: double (nullable = true)
 |-- Q_Area_II: double (nullable = true)
 |-- Q_PeakTime_II: double (nullable = true)
 |-- R_Area_II: double (nullable = true)
 |-- R_PeakTime_II: double (nullable = true)
 |-- S_Area_II: double (nullable = true)
 |-- S_PeakTime_II: double (nullable = true)
 |-- QRS_Balance_II: double (nullable = true)
 |-- T_Area_II: double (nullable = true)
 |-- T_PeakTime_II: double (nullable = true)
 |-- QRS_Are

In [10]:
ecgdeli.printSchema()

root
 |-- ecg_id: integer (nullable = true)
 |-- PQ_Int_I: double (nullable = true)
 |-- PQ_Int_I_iqr: double (nullable = true)
 |-- PQ_Int_I_count: integer (nullable = true)
 |-- PQ_Int_II: double (nullable = true)
 |-- PQ_Int_II_iqr: double (nullable = true)
 |-- PQ_Int_II_count: integer (nullable = true)
 |-- PQ_Int_III: double (nullable = true)
 |-- PQ_Int_III_iqr: double (nullable = true)
 |-- PQ_Int_III_count: integer (nullable = true)
 |-- PQ_Int_V1: double (nullable = true)
 |-- PQ_Int_V1_iqr: double (nullable = true)
 |-- PQ_Int_V1_count: integer (nullable = true)
 |-- PQ_Int_V2: double (nullable = true)
 |-- PQ_Int_V2_iqr: double (nullable = true)
 |-- PQ_Int_V2_count: integer (nullable = true)
 |-- PQ_Int_V3: double (nullable = true)
 |-- PQ_Int_V3_iqr: double (nullable = true)
 |-- PQ_Int_V3_count: integer (nullable = true)
 |-- PQ_Int_V4: double (nullable = true)
 |-- PQ_Int_V4_iqr: double (nullable = true)
 |-- PQ_Int_V4_count: integer (nullable = true)
 |-- PQ_Int_V5: do

In [11]:
unig.printSchema()

root
 |-- ecg_id: integer (nullable = true)
 |-- P_On_I: double (nullable = true)
 |-- P_On_II: double (nullable = true)
 |-- P_On_III: double (nullable = true)
 |-- P_On_aVR: double (nullable = true)
 |-- P_On_aVL: double (nullable = true)
 |-- P_On_aVF: double (nullable = true)
 |-- P_On_V1: double (nullable = true)
 |-- P_On_V2: double (nullable = true)
 |-- P_On_V3: double (nullable = true)
 |-- P_On_V4: double (nullable = true)
 |-- P_On_V5: double (nullable = true)
 |-- P_On_V6: double (nullable = true)
 |-- P_DurFull_I: double (nullable = true)
 |-- P_DurFull_II: double (nullable = true)
 |-- P_DurFull_III: double (nullable = true)
 |-- P_DurFull_aVR: double (nullable = true)
 |-- P_DurFull_aVL: double (nullable = true)
 |-- P_DurFull_aVF: double (nullable = true)
 |-- P_DurFull_V1: double (nullable = true)
 |-- P_DurFull_V2: double (nullable = true)
 |-- P_DurFull_V3: double (nullable = true)
 |-- P_DurFull_V4: double (nullable = true)
 |-- P_DurFull_V5: double (nullable = true

In [12]:
# select PTB-XL columns
ptbxl.printSchema()

root
 |-- ecg_id: integer (nullable = true)
 |-- patient_id: double (nullable = true)
 |-- age: double (nullable = true)
 |-- sex: integer (nullable = true)
 |-- height: double (nullable = true)
 |-- weight: double (nullable = true)
 |-- nurse: double (nullable = true)
 |-- site: double (nullable = true)
 |-- device: string (nullable = true)
 |-- recording_date: timestamp (nullable = true)
 |-- report: string (nullable = true)
 |-- scp_codes: string (nullable = true)
 |-- heart_axis: string (nullable = true)
 |-- infarction_stadium1: string (nullable = true)
 |-- infarction_stadium2: string (nullable = true)
 |-- validated_by: string (nullable = true)
 |-- second_opinion: string (nullable = true)
 |-- initial_autogenerated_report: string (nullable = true)
 |-- validated_by_human: string (nullable = true)
 |-- baseline_drift: string (nullable = true)
 |-- static_noise: string (nullable = true)
 |-- burst_noise: string (nullable = true)
 |-- electrodes_problems: string (nullable = true)


In [13]:
ptbxl_trim = ptbxl.drop("nurse", "site", "device",  "validated_by", "recording_date")

#### IV. JOIN ptbxl_trim to the numeric features (unig, ecgdeli, sl12)
---

In [14]:
# Join data files into ptbxl_nw (no ecg waveforms)

# adding prefix to distinguish between three different feature (algorithm) variables
def add_prefix(df, prefix, key="ecg_id"):
    return df.select(key, *[df[c].alias(f"{prefix}_{c}") for c in df.columns if c != key])

sl12_p = add_prefix(sl12, "sl12")
ecgdeli_p = add_prefix(ecgdeli, "ecgdeli")
unig_p = add_prefix(unig, "unig")

# JOIN
ptbxl_nw = ptbxl_trim.join(sl12_p, on="ecg_id", how="inner").join(ecgdeli_p, on="ecg_id", how="inner").join(unig_p, on="ecg_id", how="inner")
ptbxl_nw.show(5, truncate=True)

26/10/28 14:30:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/10/28 14:30:48 WARN DAGScheduler: Broadcasting large task binary with size 1860.2 KiB
                                                                                

+------+----------+----+---+------+------+--------------------+--------------------+----------+-------------------+-------------------+--------------+----------------------------+------------------+--------------+------------+-----------+-------------------+-----------+---------+----------+--------------------+--------------------+-------------+-----------------+--------------------+-----------------+-------------+-----------------+-------------------+-----------------+------------------+------------------+-----------------+---------------+--------------+------------------+--------------+------------------+--------------+------------------+-------------------+------------------+-------------------+--------------+------------------+------------------+--------------+------------------+--------------+------------------+-------------------+------------------+--------------+------------------+-------------------+--------------+------------------+----------------+-------------------+--------

#### Save to /standard/ ... /team2
**Note:** Parquet stores the schema inside the file

In [15]:
# save ptbxl_nw to ~/team2 folder
ptbxl_nw.write.mode("overwrite").parquet(f"{base_team2}/ptbxl_nw")

26/10/28 14:31:09 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
                                                                                